In [25]:
"""
Create another dataset from the original dataset which consist of only numerical values.
"""

import pandas as pd

file_path = "GPA_clean.xlsx"
df = pd.read_excel(file_path)

df.dropna().drop_duplicates(inplace=True)
df.drop(['Xếp loại học tập'], axis=1, inplace=True)

df.rename(columns={
    'GPA_1': 'gpa_1', 'GPA_2': 'gpa_2',
    'GPA_3': 'gpa_3', 'GPA_4': 'gpa_4',
    'GPA_5': 'gpa_5', 'GPA_6': 'gpa_6',
}, inplace=True)

output_path = "Data_GPA_clean.xlsx"
df.to_excel(output_path, index=False)

In [31]:
"""
Use the new dataset created to train an XGBoost/RandomForest model for
each grade range ('gpa_1' -> 'gpa_2'; 'gpa_1 + gpa_2' -> 'gpa_3',...).
Each model is evaluated using MSE criteria and saved using joblib.dump().
"""

import joblib
import pandas as pd
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

df = pd.read_excel("Data_GPA_clean.xlsx")

xgb_model_list, xgb_mse_list = [], []
rf_model_list, rf_mse_list = [], []
for train in range(1, 6):
    X = []
    y = []
    for row in df.values.tolist():
        X.append(row[:train])  # Previous GPA
        y.append(row[train])   # Next GPA

    # Split data into train and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # =====================
    # Train XGBoost model
    # =====================
    xgb_model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        objective='reg:squarederror',
        random_state=42,
    )
    xgb_model.fit(X_train, y_train)
    xgb_model_list.append(xgb_model)

    # Evaluate model
    y_result = xgb_model.predict(X_test)
    mse = mean_squared_error(y_test, y_result)
    xgb_mse_list.append(mse)
    print(f"XGB: GPA_{train + 1} prediction MSE: {mse:.4f}")

    # =====================
    # Train RandomForest model
    # =====================
    rf_model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)
    rf_model_list.append(rf_model)

    # Evaluate model
    y_result = rf_model.predict(X_test)
    mse = mean_squared_error(y_test, y_result)
    rf_mse_list.append(mse)
    print(f"RF: GPA_{train + 1} prediction MSE: {mse:.4f}")

mean = lambda x: sum(x) / len(x)
print(f"\nXGB average MSE: {mean(xgb_mse_list):.4f}")
print(f"RF average MSE: {mean(rf_mse_list):.4f}")

# Save the models using joblib
_ = joblib.dump(xgb_model_list, "xgb_models.joblib")
_ = joblib.dump(rf_model_list, "rf_models.joblib")

XGB: GPA_2 prediction MSE: 0.3487
RF: GPA_2 prediction MSE: 0.3454
XGB: GPA_3 prediction MSE: 0.2490
RF: GPA_3 prediction MSE: 0.2453
XGB: GPA_4 prediction MSE: 0.2344
RF: GPA_4 prediction MSE: 0.2320
XGB: GPA_5 prediction MSE: 0.1826
RF: GPA_5 prediction MSE: 0.1886
XGB: GPA_6 prediction MSE: 0.4051
RF: GPA_6 prediction MSE: 0.3887

XGB average MSE: 0.2840
RF average MSE: 0.2800
